<a href="https://colab.research.google.com/github/Roshan2008-M/CSA0909-PROGRAMME-IN-JAVA--192524057-/blob/main/ASSINGMENT_3_4_to_7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import threading
import time

class ScannerStation(threading.Thread):
    def __init__(self, name, scripts):
        super().__init__(name=name)
        self.scripts = scripts

    def run(self):
        try:
            time.sleep(1)
            print(f"{self.name} scanned {self.scripts} scripts")
        except Exception as e:
            print(f"Thread interrupted: {e}")

def main():
    stations = []
    for i in range(4):
        station = ScannerStation(name=f"Station {i + 1}", scripts=10)
        stations.append(station)
        station.start()

    for t in stations:
        t.join()

    print("All Stations Completed")

if __name__ == "__main__":
    main()

Station 1 scanned 10 scriptsStation 2 scanned 10 scripts
Station 3 scanned 10 scripts

Station 4 scanned 10 scripts
All Stations Completed


In [4]:
import threading

class SeatBooking:
    def __init__(self, seats):
        self.booked = [False] * seats
        self.bookings_made = 0
        self._lock = threading.Lock()  # Lock to synchronize access

    def book(self, seat):
        with self._lock:
            if not self.booked[seat]:
                self.booked[seat] = True
                self.bookings_made += 1

def main():
    screen = SeatBooking(10)
    users = []

    for i in range(100):
        seat = i % 10
        t = threading.Thread(target=screen.book, args=(seat,))
        users.append(t)
        t.start()

    for t in users:
        t.join()

    print(f"Seats sold: {screen.bookings_made}")

if __name__ == "__main__":
    main()

Seats sold: 10


In [6]:
import threading

class TokenCounter:
    def __init__(self):
        self.buffer = [0] * 5
        self.count = 0
        self.in_idx = 0
        self.out_idx = 0
        self.condition = threading.Condition()  # Python equivalent of Java's monitor/synchronized

    def put(self, token):
        with self.condition:
            while self.count == len(self.buffer):
                self.condition.wait()

            self.buffer[self.in_idx] = token
            self.in_idx = (self.in_idx + 1) % len(self.buffer)
            self.count += 1
            self.condition.notify_all()

    def take(self):
        with self.condition:
            while self.count == 0:
                self.condition.wait()

            token = self.buffer[self.out_idx]
            self.out_idx = (self.out_idx + 1) % len(self.buffer)
            self.count -= 1
            self.condition.notify_all()
            return token

def run_cook(counter):
    for t in range(1, 21):
        counter.put(t)

def run_student(counter, collected, student_id):
    student_sum = 0
    for _ in range(10):
        student_sum += counter.take()
    collected[student_id] = student_sum

def main():
    counter = TokenCounter()
    collected = [0, 0]

    cook = threading.Thread(target=run_cook, args=(counter,))

    students = []
    for s in range(2):
        t = threading.Thread(target=run_student, args=(counter, collected, s))
        students.append(t)

    cook.start()
    for t in students:
        t.start()

    cook.join()
    for t in students:
        t.join()

    total_collected = sum(collected)
    print(f"Sum of tokens collected: {total_collected} (expected 210)")

if __name__ == "__main__":
    main()

Sum of tokens collected: 210 (expected 210)


In [8]:
import concurrent.futures
from pathlib import Path

class FileResult:
    def __init__(self, file_path, success, score):
        self.file = file_path
        self.success = success
        self.score = score

def process(file_path):
    try:
        # Read file content and parse float
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError()

        data = path.read_text().strip()
        score = float(data)

        if score < 0 or score > 100:
            raise ValueError("Invalid Score")

        return FileResult(str(file_path), True, score)
    except Exception:
        return FileResult(str(file_path), False, 0.0)

def main():
    # Simulate files for test purposes (uncomment to create files if needed)
    # Path("marks1.txt").write_text("85")
    # Path("marks2.txt").write_text("105")  # Invalid
    # Path("marks3.txt").write_text("72")

    files = [
        Path("marks1.txt"),
        Path("marks2.txt"),
        Path("marks3.txt")
    ]

    results = []

    # Using ThreadPoolExecutor as the equivalent of ExecutorService
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(process, file): file for file in files}
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())

    success = sum(1 for r in results if r.success)
    failure = len(results) - success

    print(f"Total Files: {len(results)}")
    print(f"Successful: {success}")
    print(f"Failed: {failure}")

if __name__ == "__main__":
    main()

Total Files: 3
Successful: 0
Failed: 3
